In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random

In [2]:
fake = Faker()
np.random.seed(42)
random.seed(42)

NUM_ROWS = 50000



# ---------------------------
# Domain demand in market
# (company restructuring scenario)
# ---------------------------
domain_growth = {
    'Software Development': 0.75,
    'Data Analysis': 0.85,
    'DevOps': 0.95,
    'Quality Assurance': 0.45,
    'Technical Support': 0.40,
    'UI/UX Design': 0.65,
    'Project Management': 0.60
}

# ---------------------------
# Market demand score of skills
# ---------------------------
skill_market_value = {
    'Python': 0.95,
    'Java': 0.75,
    'Cloud Computing': 1.00,
    'Generative AI': 1.00,
    'Prompt Engineering': 0.95,
    'React': 0.85,
    'Kubernetes': 0.95,
    'SQL': 0.80,
    'Automation Testing': 0.60,
    'Manual Testing': 0.30,
    'Legacy Systems': 0.20
}

skills_list = list(skill_market_value.keys())
domains = list(domain_growth.keys())

# Base salary per domain
base_salary = {
    'Software Development': 90000,
    'Data Analysis': 85000,
    'DevOps': 120000,
    'Quality Assurance': 70000,
    'Technical Support': 60000,
    'UI/UX Design': 80000,
    'Project Management': 100000
}


In [3]:
data = []

print("Generating realistic ML dataset...")

for i in range(NUM_ROWS):

    emp_id = f"EMP{100000+i}"
    age = random.randint(22, 60)

    # Experience correlated with age
    experience = max(0, age - random.randint(21, 25))

    domain = random.choice(domains)

    # Salary depends on experience + domain
    salary = np.random.normal(
        base_salary[domain] + experience * 2500,
        15000
    )
    salary = int(max(30000, salary))

    # Select skills
    num_skills = random.randint(3, 6)
    employee_skills = random.sample(skills_list, num_skills)

    # Calculate skill demand score
    skill_score = np.mean([skill_market_value[s] for s in employee_skills])

    # Domain demand
    domain_score = domain_growth[domain]

    # Cost pressure factor (company cutting expensive employees)
    cost_factor = salary / 180000

    # Aging workforce risk
    age_factor = age / 60

    # FINAL LAYOFF PROBABILITY (NO HARD RULES)
    layoff_prob = (
        0.35 * (1 - skill_score) +
        0.25 * (1 - domain_score) +
        0.25 * cost_factor +
        0.15 * age_factor
    )

    layoff_prob = min(max(layoff_prob, 0), 1)

    layoff = 1 if np.random.rand() < layoff_prob else 0

    # Binary skill encoding (ML friendly)
    skill_features = {skill: int(skill in employee_skills) for skill in skills_list}

    row = {
        "Emp_ID": emp_id,
        "Age": age,
        "Experience": experience,
        "Domain": domain,
        "Salary": salary,
        "Skill_Score": round(skill_score,3),
        "Domain_Score": domain_score,
        "Layoff": layoff
    }

    row.update(skill_features)

    data.append(row)

df = pd.DataFrame(data)

print("\nDataset Shape:", df.shape)
print("\nLayoff Distribution:")
print(df['Layoff'].value_counts(normalize=True))

Generating realistic ML dataset...

Dataset Shape: (50000, 19)

Layoff Distribution:
Layoff
0    0.5483
1    0.4517
Name: proportion, dtype: float64


In [ ]:
df.to_parquet("/home/bhuvaneshwaran/Desktop/Medium/parquetFiles/Emp_layoff.parquet",engine="pyarrow",index=False)